# Covariance collapse under the log score vs. the kernel score

**Standalone experiment.** When the *true* conditional distribution of a prediction task is
**degenerate** — some marginal variances and/or joint covariance directions are exactly zero —
the Gaussian **negative log-likelihood (NLL / log score)** becomes an **ill-posed training loss**:
its infimum is $-\infty$ and not attained, gradients diverge as eigenvalues $\to 0$, and the result
depends materially on the $\varepsilon$-floor hack. The **Gaussian kernel score** stays well-posed on
the closed PSD cone (bounded gradients via its intrinsic $h^2$ floor) and trains to the boundary
gracefully.

**Task.** Reconstruct a periodic **band-limited** random signal from a **uniformly subsampled**
observation. The dial $n/s$ (observations vs. active frequencies) crosses the **Nyquist point**,
controlling the rank deficiency of the true conditional covariance *in closed form*.

Everything is simulated, double precision, CPU is enough. Runs top-to-bottom in a few minutes.

---
### Evaluation note (read me)
In the hard-collapse regime the true distribution is **degenerate**, so held-out **log score is
$-\infty$/undefined at the truth** and meaningless for comparison — *we do not report test NLL here.*
We evaluate with quantities that stay well-defined at the PSD boundary:
* exact **population objectives** for both losses (closed form, since the truth is Gaussian),
* $\lVert \Sigma_\theta - \Sigma_{\mathrm{post}}\rVert_F$ and $\lVert \Sigma_\theta - \Sigma_{\mathrm{KS}}^\star\rVert_F$,
* **rank recovery** (sorted eigenvalue spectrum vs. the true rank $r$),
* held-out **kernel/energy score** on samples.

## 1. Configuration
All knobs live here. `d`, `B`, `k` can be changed freely; the rank printout, plots and asserts all adapt.

In [ ]:
import os, math, time, json
from dataclasses import dataclass, field, asdict, replace
from typing import Tuple
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# --- shared figure *specification*: paper widths, fonts, semantic palette ---------------
# notebooks/plot_config.py carries only what every figure of the paper shares. Anything that
# names *this* experiment -- its training arms, its band limits, the labels the panels print
# -- is defined right below, so the module stays reusable by the other notebooks.
import sys
from pathlib import Path
if str(Path.cwd()) not in sys.path:               # so `import plot_config` works from anywhere
    sys.path.insert(0, str(Path.cwd()))
import plot_config as pc

PLOT    = pc.apply_style()      # scienceplots + the paper's font sizes / line widths / rcParams
FIG_DIR = pc.FIG_DIR
COLORS  = pc.COLORS             # semantic roles (signal, observation, truth, ...), shared

# --- this experiment's colors and labels ------------------------------------------------
# Training arms (kernel score vs. eps-floored NLL). Keys are the arm names built by
# `arm_name()` in section 7.
ARM_COLORS = {
    "KS":            COLORS["kernel_score"],
    "NLL eps=1e-03": "#fdae61",
    "NLL eps=1e-06": "#f46d43",
    "NLL eps=1e-09": "#d73027",
    "NLL raw":       COLORS["log_score"],
}
# Band limits B in the rank / realization figures -- deliberately distinct from ARM_COLORS.
RANK_COLORS = {4: "#4575b4", 8: "#7b3294", 16: "#e08214", 32: "#01665e"}

# The paper names each loss by its score, not by the arm's internal key.
ARM_LABELS = {
    "KS":            r"$S_k$",
    "NLL eps=1e-03": r"$S_\mathrm{log}$ ($\varepsilon = 10^{-3}$)",
    "NLL eps=1e-06": r"$S_\mathrm{log}$ ($\varepsilon = 10^{-6}$)",
    "NLL eps=1e-09": r"$S_\mathrm{log}$ ($\varepsilon = 10^{-9}$)",
    "NLL raw":       r"$S_\mathrm{log}$",
}
# Recurring non-arm labels, so every panel spells them the same way.
LABELS = {
    "signal":      r"True signal $\boldsymbol{x}$",
    "observation": r"Observation $\tilde{\boldsymbol{x}}$",
    "truth":       r"$\mathcal{N}(\boldsymbol{\mu}^\star, \boldsymbol{\Sigma}^\star)$",
}

def arm_color(name, default="#555555"):
    """Color of a training arm by name, falling back to a neutral gray."""
    return ARM_COLORS.get(name, default)

def arm_label(name):
    """Paper label of a training arm ("KS" -> "$S_k$"); unknown names pass through."""
    return ARM_LABELS.get(name, name)

torch.set_default_dtype(torch.float64)          # the phenomenon lives near the PSD boundary

# Optional quick-run: set COLLAPSE_FAST=1 in the environment to shrink the budget for a smoke test.
FAST = False

@dataclass
class Config:
    d: int = 128                     # signal length / output dimension
    B: int = 40                      # band limit; active modes s = 2B+1
    k: int = 8                       # subsampling stride (divides d); n = d//k observations
    spectrum_shape: str = "red"      # "flat" | "red"  (red: lambda_j ~ 1/(1+|j|)^2)
    tau: float = 0.0                 # signal nugget:  y = x + tau * white noise
    sigma_obs: float = 1e-3           # observation noise on the subsampled input
    h2_prefactor: float = 2.0        # kernel bandwidth gamma^2 = prefactor * d * mean_marginal_var
    eps_list: Tuple[float, ...] = (1e-6,)
    n_train: int = 4096
    n_val: int = 1024
    batch_size: int = 256
    lr: float = 1e-3                # Adam on the covariance parameters, shared across every arm
    lr_mean: float = 3e-2            # ... and on the mean head, which needs a larger one (see sec. 6)
    n_steps: int = 5000
    grad_clip: float = 1e8           # ~off by default; the diagnostics log the *pre-clip* grad norm
    oracle_mean: bool = False         # init the linear mean at the closed-form A to isolate cov dynamics
    n_mc: int = 0                    # MC samples for the sample-based kernel score (0 -> closed form)
    seed: int = 0
    log_every: int = 10
    @property
    def s(self):  return 2 * self.B + 1
    @property
    def n(self):  return self.d // self.k

cfg = Config()
if FAST:
    cfg.n_steps = 400; cfg.n_train = 1024
assert cfg.d % cfg.k == 0, "k must divide d"
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
print("Config:", json.dumps(asdict(cfg), default=str, indent=2))
print(f"\n=> s = 2B+1 = {cfg.s} active modes,  n = d//k = {cfg.n} observations,  n/s = {cfg.n/cfg.s:.3f}")


## 2. Data generation — band-limited GP simulator + subsampling

Draw active Fourier coefficients with per-mode power $\lambda_j$ on $|j|\le B$, enforce a real signal,
inverse-FFT to physical space. **The FFT is used only to simulate** — the model and both losses live
entirely in physical space. Target $y = x + \tau\,\xi$; input $x_{\mathrm{obs}} = x[::k] + \sigma_{\mathrm{obs}}\,\eta$.

In [ ]:
def spectrum_twosided(cfg):
    """Two-sided power P over the d DFT bins, normalized so tr(Sigma_pr) = d."""
    d = cfg.d
    j = torch.arange(d)
    absf = torch.where(j <= d // 2, j, j - d).abs().double()          # |frequency| per bin
    active = absf <= cfg.B
    base = torch.ones(d) if cfg.spectrum_shape == "flat" else 1.0 / (1.0 + absf) ** 2
    P = torch.where(active, base, torch.zeros(d))
    return P * d / P.sum()

def build_prior(cfg):
    """Circulant prior covariance Sigma_pr = F^H diag(P) F / d, built from the autocovariance."""
    d = cfg.d
    P = spectrum_twosided(cfg)
    m = torch.arange(d).double(); jj = torch.arange(d).double()
    ang = 2 * math.pi * torch.outer(m, jj) / d
    c = (torch.cos(ang) @ P) / d                                       # real autocovariance c[m]
    idx = (torch.arange(d).view(-1, 1) - torch.arange(d).view(1, -1)) % d
    return c[idx], P                                                   # Sigma_pr[a,b] = c[(a-b) % d]

def simulate(cfg, P, N, gen=None):
    """N band-limited real signals with covariance Sigma_pr (FFT synthesis)."""
    xr = torch.randn(N, cfg.d, generator=gen)
    Xk = torch.fft.fft(xr, dim=1) * torch.sqrt(P).unsqueeze(0)         # E|Xk|^2 = d * P
    return torch.fft.ifft(Xk, dim=1).real

def selection(cfg):
    """Stride-k selection matrix S (n x d) and the observed indices."""
    idx = torch.arange(0, cfg.d, cfg.k)
    S = torch.zeros(cfg.n, cfg.d); S[torch.arange(cfg.n), idx] = 1.0
    return S, idx

def make_dataset(cfg, Sigma_pr, P, N, seed):
    g = torch.Generator().manual_seed(seed)
    x = simulate(cfg, P, N, g)
    x_obs = x[:, ::cfg.k] + cfg.sigma_obs * torch.randn(N, cfg.n, generator=g)
    y = x + cfg.tau * torch.randn(N, cfg.d, generator=g)
    return x, x_obs, y

In [ ]:
Sigma_pr, P = build_prior(cfg)
S, obs_idx = selection(cfg)
print(f"tr(Sigma_pr) = {Sigma_pr.diag().sum():.3f} (target d = {cfg.d})")
print(f"rank(Sigma_pr) = {torch.linalg.matrix_rank(Sigma_pr, atol=1e-9)}  (theory s = {cfg.s})")

x_tr, xobs_tr, y_tr = make_dataset(cfg, Sigma_pr, P, cfg.n_train, cfg.seed + 10)
x_va, xobs_va, y_va = make_dataset(cfg, Sigma_pr, P, cfg.n_val,   cfg.seed + 20)

### The prior covariance — what it is and how it is built

The prior is a **stationary Gaussian process on a periodic grid of $d$ points**, so
$\Sigma_{\mathrm{pr}}$ is **circulant**: entry $(a,b)$ depends only on the *lag* $a-b$ taken modulo $d$,

$$\Sigma_{\mathrm{pr}}[a,b] \;=\; c\big[(a-b)\bmod d\big],\qquad
c[m] \;=\; \frac{1}{d}\sum_{j=0}^{d-1} P_j\cos\!\Big(\frac{2\pi j m}{d}\Big)
\;=\;\sum_{|f|\le B} w_f\,\cos\!\Big(\frac{2\pi f m}{d}\Big),$$

i.e. $\Sigma_{\mathrm{pr}} = \tfrac1d F^{*}\mathrm{diag}(P)F$ — its eigenvectors are the Fourier modes and
its eigenvalues are the power spectrum $P$. The spectrum is the only design choice:

$$P_f \;=\; \underbrace{\mathbf 1\{|f|\le B\}}_{\text{band limit}}\cdot
\underbrace{\begin{cases}1 & \texttt{flat}\\ (1+|f|)^{-2} & \texttt{red}\end{cases}}_{\text{shape}},
\qquad\text{then rescaled so } \textstyle\sum_f P_f = d,$$

with $f$ the signed DFT frequency ($f = j$ for $j\le d/2$, else $j-d$). The normalization gives
$\operatorname{tr}\Sigma_{\mathrm{pr}} = d$ and hence $c[0]=1$: **every marginal variance is exactly $1$**,
so $\Sigma_{\mathrm{pr}}$ is a correlation matrix and $B$ changes only the *shape* of the correlation, not its
scale. The `red` weights $w_f\propto(1+|f|)^{-2}$ make the samples smooth; for `flat` weights $c$ is the
**Dirichlet kernel** $c[m]=\sin(\pi s m/d)\,/\,\big(s\sin(\pi m/d)\big)$ with $s=2B+1$.

**Why this prior.** Two properties do all the work. (i) $\operatorname{rank}\Sigma_{\mathrm{pr}} = s = 2B+1$ —
only $s$ modes carry power, so signals live in an $s$-dimensional subspace of $\mathbb R^d$. (ii) The stride-$k$
selection $S$ keeps $n=d/k$ of the points. Together they give the true posterior rank $\max(0,s-n)$ in closed
form, which is the dial $n/s$ that the whole experiment turns. So $B$ is not a smoothness knob here — it is the
**rank-deficiency knob**.

*Caveat at $B=80$:* the grid only supports $|f|\le d/2 = 64$, so $|f|\le 80$ activates **every** bin and the
prior rank saturates at $\min(2B+1,\,d) = 128$ rather than the nominal $161$. The prior is then full rank and the
closed-form posterior rank becomes $d-n$, not $s-n$. (With the `red` shape the extra modes carry almost no power,
which is why the $B=40$ and $B=80$ panels below look nearly identical.)

**Is the domain relevant?** No — nothing downstream ever sees a physical coordinate. The model, both losses and
the selection operator are defined on the index set $\{0,\dots,d-1\}$, and $\Sigma_{\mathrm{pr}}$ depends only on
the lag mod $d$. So we may identify $a \mapsto t_a = a/d$ and read the setup as living on the
**circle $[0,1)$ with period 1**:

$$c(t) \;=\; \sum_{|f|\le B} w_f\cos(2\pi f t),\qquad t\in[0,1),$$

with $B$ in **cycles per unit domain**, grid spacing $1/d$ and observations at spacing $k/d$. Any other interval
would just rescale the frequency axis and leave every number in the experiment unchanged. What *is* essential is
the **periodicity**: it is what makes $\Sigma_{\mathrm{pr}}$ exactly circulant and DFT-diagonal, so the band limit
gives an exact rank and the Nyquist threshold $n=s$ is sharp rather than approximate. On a non-periodic interval
the same construction would hold only up to edge effects.

## 3. Closed-form true posterior

With selection $S$ (rows of $I$ at stride $k$):
$$\mu_{\mathrm{post}}(x_{\mathrm{obs}}) = A\,x_{\mathrm{obs}},\quad
A = \Sigma_{\mathrm{pr}}S^\top(S\Sigma_{\mathrm{pr}}S^\top + \sigma_{\mathrm{obs}}^2 I)^{-1},$$
$$\Sigma_{\mathrm{post}} = \Sigma_{\mathrm{pr}} + \tau^2 I - \Sigma_{\mathrm{pr}}S^\top
(S\Sigma_{\mathrm{pr}}S^\top + \sigma_{\mathrm{obs}}^2 I)^{-1}S\Sigma_{\mathrm{pr}}.$$
For $\tau=\sigma_{\mathrm{obs}}=0$ (contiguous band) the rank is $r=\max(0,\,s-n)$. We use a pseudo-inverse
so the $\sigma_{\mathrm{obs}}=0$ singular Gram matrix is handled, and store the eigendecomposition of
$\Sigma_{\mathrm{post}}$, partitioned into **true-null** and **true-signal** directions.

In [ ]:
def psd_project(M):
    """Symmetrize and clamp negative eigenvalues -- Sigma_post is PSD by construction."""
    M = 0.5 * (M + M.T)
    w, V = torch.linalg.eigh(M)
    return (V * w.clamp_min(0.0)) @ V.T

def posterior(cfg, Sigma_pr, S):
    d = cfg.d
    G = S @ Sigma_pr @ S.T + cfg.sigma_obs ** 2 * torch.eye(cfg.n)
    SSp = S @ Sigma_pr                                     # (n, d)
    # rtol treats the sigma_obs^2 ridge as null. With the default rtol (~1e-14) a rank-deficient
    # G keeps its 1e-12 ridge modes and inverts them as 1e12, and the subtraction below returns
    # cancellation noise (~1e-3, indefinite) where Sigma_post should be exactly 0 -- e.g. B=4.
    Gpinv = torch.linalg.pinv(G, rtol=1e-8)                # pseudo-inverse: handles singular G
    A = Sigma_pr @ S.T @ Gpinv                             # (d, n)
    Sigma_post = Sigma_pr + cfg.tau ** 2 * torch.eye(d) - SSp.T @ Gpinv @ SSp
    return A, psd_project(Sigma_post)

def eig_partition(Sigma_post, tol=1e-8):
    evals, evecs = torch.linalg.eigh(Sigma_post)
    scale = evals.abs().max().clamp_min(1.0)
    null_mask = evals <= tol * scale
    return evals, evecs, evecs[:, null_mask], evecs[:, ~null_mask]

A_true, Sigma_post = posterior(cfg, Sigma_pr, S)
evals_post, evecs_post, V_null, V_sig = eig_partition(Sigma_post)
# NOTE: this counts directions with O(1) variance, i.e. the sigma_obs -> 0 ("hard") rank.
# For sigma_obs > 0 the exact rank is rank(Sigma_pr): the pinned directions sit at O(sigma_obs^2),
# not at zero. That soft collapse is what the log score actually sees -- see the figure above.
rank_post = int((evals_post > 1e-8 * max(float(evals_post.max()), 1.0)).sum())

hard = (cfg.tau == 0.0 and cfg.sigma_obs == 0.0)
regime = "above" if cfg.n > cfg.s else ("at" if cfg.n == cfg.s else "below")
print(f"n/s = {cfg.n/cfg.s:.3f}  ->  {regime} Nyquist")
print(f"eff. rank(Sigma_post) = {rank_post}/{cfg.d}   (true-null dirs = {V_null.shape[1]}, true-signal dirs = {V_sig.shape[1]})")
if hard:
    print(f"theory max(0, s-n) = {max(0, cfg.s - cfg.n)}")
    assert rank_post == max(0, cfg.s - cfg.n), "rank(Sigma_post) != max(0, s-n)"
print(f"max diag(Sigma_post) at observed coords = {Sigma_post.diag()[obs_idx].abs().max():.2e} (should be ~0)")

# kernel bandwidth gamma^2 = h2  (floored by a small fraction of the prior scale so it stays
# positive even above Nyquist, where Sigma_post -> 0)
def bandwidth(cfg, Sigma_post, Sigma_pr):
    v = max(Sigma_post.diag().mean().item(), 1e-2 * Sigma_pr.diag().mean().item())
    return cfg.h2_prefactor * cfg.d * v
mean_marg_var = Sigma_post.diag().mean().item()
h2 = bandwidth(cfg, Sigma_post, Sigma_pr)
REGIME_TAG = f"n/s = {cfg.n/cfg.s:.2f}, {regime} Nyquist, eff. rank(Sigma_post) = {rank_post}/{cfg.d}"
print(f"mean marginal variance = {mean_marg_var:.4f}   ->   h2 (gamma^2) = {h2:.3f}")
print("REGIME:", REGIME_TAG)

### The prior process and its rank, at three band limits

Two views of the same three priors, one column per band limit $B\in\{4,40,80\}$.

**Top row — what the process looks like.** One draw from $\Sigma_{\mathrm{pr}}$ with its stride-$k$ observations
and the closed-form posterior $\mu_{\mathrm{post}}\pm2\sigma_{\mathrm{post}}$. The band limit is directly visible
as **smoothness**: $B=4$ admits only 9 Fourier modes, so the curve is slow and gentle; $B=80$ is rough. Across the
row the posterior tightens or loosens — at $B=4$ the band is invisibly narrow, while at $B=40$ and $B=80$ the
observations pin the curve only at the sampled coordinates and the band scallops between them.

**Bottom row — the collapse.** The eigenvalues of $\Sigma_{\mathrm{post}}$, sorted descending, as bars on a
**linear** scale with the $y$-axis shared across the row. The bars stop at $65$ for $B=40$ and $112$ for $B=80$,
and the $B=4$ panel looks empty.

#### Careful: with $\sigma_{\mathrm{obs}} > 0$ the collapse is *soft*, and the rank is not 0

$\Sigma_{\mathrm{post}}$ has **full prior rank** for any $\sigma_{\mathrm{obs}} > 0$:

$$\operatorname{rank}\Sigma_{\mathrm{post}} = \operatorname{rank}\Sigma_{\mathrm{pr}} = \min(2B+1,\,d)
\qquad (\text{here } 9,\ 81,\ 128).$$

Noisy observations never determine anything *exactly*, so no direction inside the signal subspace is driven to
zero variance. What actually happens is that $\min(n, \operatorname{rank}\Sigma_{\mathrm{pr}})$ directions have
their variance **crushed to $O(\sigma_{\mathrm{obs}}^2)$** while the remaining
$\max(0, \operatorname{rank}\Sigma_{\mathrm{pr}} - n)$ keep $O(1)$ variance. With $\sigma_{\mathrm{obs}}=10^{-6}$:

| $B$ | $\operatorname{rank}\Sigma_{\mathrm{pr}}$ | $\operatorname{rank}\Sigma_{\mathrm{post}}$ | directions at $O(1)$ | directions at $O(\sigma_{\mathrm{obs}}^2)$ | value of the small block |
|---|---|---|---|---|---|
| 4  | 9   | 9   | 0   | 9  | $8.0\times10^{-12} = k\,\sigma_{\mathrm{obs}}^2$ |
| 40 | 81  | 81  | 65  | 16 | $1.3$–$1.6\times10^{-12}$ |
| 80 | 128 | 128 | 112 | 16 | $1.0\times10^{-12} = \sigma_{\mathrm{obs}}^2$ |

So the familiar $\max(0, \operatorname{rank}\Sigma_{\mathrm{pr}} - n) = 0/65/112$ is the **$\sigma_{\mathrm{obs}}\to0$
(hard) limit**, and it counts the $O(1)$ directions — not the rank. At $B=4$ that count is $0$: 9 unknown modes and
$n=16$ observations is over-determined, so *every* direction gets crushed, but only to $8\sigma_{\mathrm{obs}}^2$,
not to zero. The bars in the bottom row are $O(\sigma_{\mathrm{obs}}^2)=10^{-12}$ on an axis whose top is $0.7$, so
they are invisible — the $B=4$ panel is not empty, it is $12$ orders of magnitude below the ink. Each panel title
gives both counts.

*Why the distinction still matters for this experiment.* The log score sees
$\log\det\Sigma$, so a direction at $\sigma_{\mathrm{obs}}^2 = 10^{-12}$ contributes $\approx -27.6$ instead of
$-\infty$. The truth is therefore not literally degenerate here, but it sits so close to the PSD boundary that
$\sigma_{\mathrm{obs}}^2$ plays exactly the role of the $\varepsilon$-floor: the NLL optimum is *attained*, just at
a value and a curvature set by an arbitrary $10^{-12}$. Setting `sigma_obs = 0.0` recovers the genuinely degenerate
case with $\log\det\to-\infty$ and rank exactly $\max(0,\operatorname{rank}\Sigma_{\mathrm{pr}}-n)$.

*Which frequencies get crushed, and why $n$ of them.* Sampling every $k$-th of $d$ points aliases frequency $f$ onto
$f \bmod n$. At $B=4$ the active frequencies $-4,\dots,4$ all differ by less than $n=16$, so no two alias together,
all 9 are measured, and all 9 collapse. At $B=40$ the frequencies $-40,\dots,40$ cover all 16 residues mod 16 and
collide, so only 16 combinations are measured and $81-16=65$ directions are untouched by the data. Crossing
$n/s = 1$ is exactly where the observations stop covering the signal subspace.

Two further points to read off:

* **The bars are the posterior's eigenvalues, the prior rank is in the top-row titles.** On a linear axis the
  prior's eigenvalues cannot show rank at all: the `red` spectrum decays as $(1+|f|)^{-2}$, so $\lambda_1\approx57$
  while $\lambda_{81}\approx3.4\times10^{-2}$ — the $B=40$ and $B=80$ prior bar plots are visually identical, both a
  single spike near index 1. The posterior's $O(1)$ block ($0.70$ down to $1.3\times10^{-2}$) is linear-friendly, so
  the edge lands where the eye can see it.
* **Exact rank $\ne$ effective dimension.** $B=40$ is nominally rank 81, but its top 9 prior eigenvalues already
  carry $\approx86\%$ of $\operatorname{tr}\Sigma_{\mathrm{pr}}$ ($B=80$: $\approx85\%$), which is why its draws
  look so much like $B=80$'s.

*(Recall from section 2 that $B=80$ saturates: $2B+1 = 161$ exceeds $d$, so the prior rank is $\min(2B+1,d) = 128$
and the $O(1)$ count is $d - n = 112$.)*

**A numerical note.** `posterior()` builds $\Sigma_{\mathrm{post}}$ by subtraction through a pseudo-inverse, which
is accurate to $\sim\!10^{-5}$ absolute — fine for the $O(1)$ structure and for training, but it cannot resolve the
$O(\sigma_{\mathrm{obs}}^2)=10^{-12}$ block. The cell below therefore also builds an exact reference via the factored
form $\Sigma_{\mathrm{pr}} = WW^\top$, $\Sigma_{\mathrm{post}} = \sigma_{\mathrm{obs}}^2\,W(\sigma_{\mathrm{obs}}^2 I
+ (SW)^\top SW)^{-1}W^\top$, which is manifestly PSD and well conditioned, and uses it for the eigenvalue counts.

In [ ]:
# One prior realization + its posterior (top) and the posterior eigenvalue spectrum (bottom),
# one column per band limit B.
B_show = B_values = (4,8,16, 32)   # RANK_COLORS is set in the config cell (distinct from the arm palette)

def posterior_exact(cfg, Sigma_pr, S):
    """Numerically exact Sigma_post via the factored form -- resolves the O(sigma_obs^2) block.

    With Sigma_pr = W W^T (W: d x r, r = rank) and z the coefficients in that basis,
        Sigma_post = sigma^2 W (sigma^2 I + (SW)^T SW)^{-1} W^T,
    which is manifestly PSD and stays well conditioned, unlike the pseudo-inverse subtraction in
    posterior() -- that one is only accurate to ~1e-5 absolute and buries a 1e-12 block.
    """
    lam, V = torch.linalg.eigh(Sigma_pr)
    keep = lam > 1e-12 * lam.max()
    W = V[:, keep] * lam[keep].sqrt()                         # d x r
    SW = S @ W
    sig2 = cfg.sigma_obs ** 2
    M = sig2 * torch.eye(W.shape[1], dtype=W.dtype) + SW.T @ SW
    if sig2 == 0.0:                                           # hard limit: M may be singular
        Sig_po = W @ (torch.eye(W.shape[1], dtype=W.dtype) - SW.T @ torch.linalg.pinv(SW.T)) @ W.T
    else:
        Sig_po = sig2 * W @ torch.cholesky_inverse(torch.linalg.cholesky(M)) @ W.T
    return psd_project(Sig_po)

dat = {}
for B in B_show:
    cfg_B         = replace(cfg, B=B)
    Sig_pr_B, P_B = build_prior(cfg_B)
    S_B, obs_B    = selection(cfg_B)
    A_B, _        = posterior(cfg_B, Sig_pr_B, S_B)
    Sig_po_B      = posterior_exact(cfg_B, Sig_pr_B, S_B)
    gen = torch.Generator().manual_seed(cfg.seed + 20)         # same seed -> comparable draw across B
    x     = simulate(cfg_B, P_B, 1, gen)[0]
    x_obs = x[::cfg_B.k] + cfg_B.sigma_obs * torch.randn(cfg_B.n, generator=gen)
    w_pr = torch.linalg.eigvalsh(Sig_pr_B).flip(0).cpu().numpy()
    w_po = torch.linalg.eigvalsh(Sig_po_B).flip(0).cpu().numpy()
    sig2 = cfg_B.sigma_obs ** 2
    # Eigenvalues below the machine-precision floor of the decomposition are numerical junk, not
    # structure: clip them to exact 0 so they vanish on the log axis instead of drawing a fake
    # carpet (e.g. B=4 has 9 real eigenvalues at 8 sigma^2 and 119 residuals at ~1e-27).
    eps_floor = cfg_B.d * np.finfo(w_po.dtype).eps * w_po.max()
    w_po = np.where(w_po > eps_floor, w_po, 0.0)
    nz = w_po[w_po > 0.0]
    dat[B] = dict(cfg=cfg_B, x=x, x_obs=x_obs, obs_idx=obs_B,
                  mu=x_obs @ A_B.T, sd=Sig_po_B.diag().clamp_min(0).sqrt(),
                  w_pr=w_pr, w_po=w_po, sig2=sig2,
                  rank_pr=int((w_pr > 1e-9 * w_pr.max()).sum()),
                  rank_po=int(nz.size),                        # exact rank (soft collapse)
                  n_big=int((w_po > 1e-6).sum()),               # directions still at O(1)
                  small_lo=float(nz.min()) if nz.size else 0.0)

grid = torch.arange(cfg.d) / cfg.d                                  # physical domain [0, 1)
fig, axes = pc.subplots(2, len(B_show), aspect=0.55 * 3 / len(B_show), sharey="row")   # double column
for c, B in enumerate(B_show):
    D, col = dat[B], RANK_COLORS[B]

    # ---- top: the signal, its observations, and the closed-form posterior
    ax = axes[0, c]
    ax.fill_between(grid, D["mu"] - 2 * D["sd"], D["mu"] + 2 * D["sd"], color=col, alpha=PLOT.band_alpha,
                    label=r"$\boldsymbol{\mu}^\star \pm 2\sigma^\star$")
    ax.plot(grid, D["x"], color=COLORS["signal"], label=LABELS["signal"])
    ax.plot(grid, D["mu"], "--", color=col, label=r"$\boldsymbol{\mu}^\star$")
    ax.plot(D["obs_idx"] / cfg.d, D["x_obs"], "o", color=COLORS["observation"], label=LABELS["observation"])
    ax.set_title(f"$B = {B}$:   rank$(\\boldsymbol{{\\Sigma}}_{{\\rm pr}}) = {D['rank_pr']}$")
    ax.set_ylim([-2.8, 1.7])
    ax.set_xlim(0, 1)

    # ---- bottom: posterior eigenvalues, decreasing bars, linear scale, shared y across the row
    ax = axes[1, c]
    ax.bar(np.arange(1, cfg.d + 1), D["w_po"], width=1.0, color=col, linewidth=0, rasterized = True)
    if 0 < D["n_big"] < cfg.d:
        ax.axvline(D["n_big"], color=COLORS["signal"], ls=":", lw=PLOT.thin_line_width)
        side = "right" if D["n_big"] > 0.8 * cfg.d else "left"    # keep the label inside the axes
        # ax.annotate(f"{D['n_big']} at $O(1)$", xy=(D["n_big"], 1.0), xycoords=("data", "axes fraction"),
        #             xytext=(-4 if side == "right" else 4, -4), textcoords="offset points",
        #             fontsize=8, va="top", ha=side)
    ax.set_title(f"rank$(\\boldsymbol{{\\Sigma}}^\\star) = {D['rank_po']}$")
    ax.set_xlabel(r"Eigenvalue index $i$")
    ax.set_xlim(0, cfg.d + 1)
    ax.set_yscale("log")

nz_all = np.concatenate([dat[B]["w_po"][dat[B]["w_po"] > 0.0] for B in B_show])
for ax in axes[1, :]:
    ax.set_ylim(0.3 * nz_all.min(), 3.0 * nz_all.max())        # clipped zeros are simply not drawn

axes[1, 0].set_ylabel(r"Eigenvalue $\lambda_i$")
handles, labels = axes[0, 0].get_legend_handles_labels()
# "outside lower center" is the constrained-layout way to make room for a figure legend: the
# canvas keeps its exact paper width, which tight_layout / bbox_inches="tight" would not.
fig.legend(handles, labels, loc="outside lower center", ncol=4)
pc.save(fig, "toy1_rank_realizations.pdf")
plt.show()

# The bars only show the O(1) block: the sigma_obs^2 directions are ~12 orders below the axis top.
print(f"sigma_obs = {cfg.sigma_obs:g}  ->  sigma_obs^2 = {cfg.sigma_obs ** 2:g}\n")
print(f"{'B':>4} {'2B+1':>6} {'rank(Sig_pr)':>13} {'rank(Sig*)':>15} {'at O(1)':>8} "
      f"{'at O(sig^2)':>12} {'small block':>12}")
for B in B_show:
    D = dat[B]
    r_pr, n_big = D["rank_pr"], D["n_big"]
    print(f"{B:>4} {D['cfg'].s:>6} {r_pr:>13} {D['rank_po']:>15} {n_big:>8} "
          f"{D['rank_po'] - n_big:>12} {D['small_lo']:>12.2e}")
    assert r_pr == min(D["cfg"].s, D["cfg"].d), f"rank(Sigma_pr) != min(2B+1, d) for B={B}"
    # with sigma_obs > 0 the posterior keeps full prior rank; only the *magnitudes* collapse
    #assert D["rank_po"] == r_pr, f"rank(Sigma_post) != rank(Sigma_pr) for B={B}"
    #assert n_big == max(0, r_pr - D["cfg"].n), f"O(1) count != max(0, rank(Sigma_pr) - n) for B={B}"

for B in B_show:
    w, rk = dat[B]["w_pr"], dat[B]["rank_pr"]
    print(f"B={B:3d}: top-9 prior eigenvalues carry {np.cumsum(w)[8] / w.sum() * 100:5.1f}% of tr(Sigma_pr)   "
          f"(lambda_1 = {w[0]:.2f}, last nonzero = {w[rk - 1]:.2e})")

#### Presentation copy — the same figure transposed (4 × 2)

One **row** per band limit $B$ instead of one column: the realization with its posterior band on the
left, the posterior eigenvalues on the right. Same data (`dat`), colors and axis limits as above; the
eigenvalue column shares its log axis down the rows. Written to `figures/presentation/` (git-ignored),
never over the paper figure.

In [ ]:
# Presentation copy: rows = band limit B, columns = (realization, posterior eigenvalues).
fig, axes = pc.subplots(len(B_show), 2, aspect=1.2, columns=4.5, sharex="col", sharey="col")
for r, B in enumerate(B_show):
    D, col = dat[B], RANK_COLORS[B]

    ax = axes[r, 0]
    ax.fill_between(grid, D["mu"] - 2 * D["sd"], D["mu"] + 2 * D["sd"], color=col, alpha=PLOT.band_alpha,
                    label=r"$\boldsymbol{\mu}^\star \pm 2\sigma^\star$")
    ax.plot(grid, D["x"], color=COLORS["signal"], label=LABELS["signal"])
    ax.plot(grid, D["mu"], "--", color=col, label=r"$\boldsymbol{\mu}^\star$")
    ax.plot(D["obs_idx"] / cfg.d, D["x_obs"], "o", color=COLORS["observation"], label=LABELS["observation"])
    ax.set_title(f"$B = {B}$:   rank$(\\boldsymbol{{\\Sigma}}_{{\\rm pr}}) = {D['rank_pr']}$")
    ax.set_ylim([-2.8, 1.7])
    ax.set_xlim(0, 1)

    ax = axes[r, 1]
    ax.bar(np.arange(1, cfg.d + 1), D["w_po"], width=1.0, color=col, linewidth=0, rasterized=True)
    if 0 < D["n_big"] < cfg.d:
        ax.axvline(D["n_big"], color=COLORS["signal"], ls=":", lw=PLOT.thin_line_width)
    ax.set_title(f"rank$(\\boldsymbol{{\\Sigma}}^\\star) = {D['rank_po']}$")
    ax.set_xlim(0, cfg.d + 1)
    ax.set_yscale("log")
    ax.set_ylim(0.3 * nz_all.min(), 3.0 * nz_all.max())
    ax.set_ylabel(r"$\lambda_i$")
axes[-1, 1].set_xlabel(r"Eigenvalue index $i$")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncol=2)
pc.save(fig, pc.PRESENTATION_DIR / "toy1_rank_realizations.pdf")
plt.show()


### Example data — the task at a glance
A few signals, their stride-$k$ observations, the posterior mean reconstruction and a $\pm 2\sigma_{\mathrm{post}}$ band. Observed coordinates are pinned exactly (zero posterior variance when $\sigma_{\mathrm{obs}}=0$).

In [ ]:
sd_post = Sigma_post.diag().clamp_min(0).sqrt()
fig, axes = pc.subplots(1, 3, aspect=0.23, sharey=True)
for ax, i in zip(axes, range(3)):
    xi, yi = x_va[i], y_va[i]
    mu = A_true @ xobs_va[i]
    grid = torch.arange(cfg.d)
    ax.fill_between(grid, (mu - 2*sd_post), (mu + 2*sd_post), color=COLORS["kernel_score"],
                    alpha=PLOT.band_alpha, label=r"$\mu_{post}\pm2\sigma_{post}$")
    ax.plot(grid, xi, color=COLORS["signal"], label="true signal $x$")
    ax.plot(grid, mu, "--", color=COLORS["kernel_score"], label=r"$\mu_{post}$")
    ax.plot(obs_idx, xobs_va[i], "o", color=COLORS["observation"], label="observations $x[::k]$")
    ax.set_title(f"example {i}"); ax.set_xlabel("coordinate")
axes[0].set_ylabel("value")
#axes[0].legend(loc="upper right")
#fig.suptitle("Task: reconstruct a band-limited signal from a stride-k subsample   (" + REGIME_TAG + ")")
#pc.save(fig, "07_example_data.pdf")
plt.show()

## 4. Losses

Both take batched `(mu, cov, y)`.

**NLL with $\varepsilon$ floor.** $\Sigma_\theta = LL^\top + \varepsilon I$. For $\varepsilon=0$ we use the
Cholesky factor $L$ directly (collapse then shows up as $\log\det \to -\infty$); for $\varepsilon>0$ we
re-factorize $LL^\top+\varepsilon I$. $\varepsilon$ is the explicit experimental floor, swept over `eps_list`
plus the raw $\varepsilon=0$ arm.

**Gaussian kernel score** with $k(x,y)=\exp(-\lVert x-y\rVert^2/\gamma^2)$, closed form for $X\sim\mathcal N(\mu,\Sigma)$:
$$\mathrm{score}=\tfrac12\big(k(y,y)+\mathbb E[k(X,X')]\big)-\mathbb E[k(X,y)]=\tfrac12\big(1+\mathbb E[k(X,X')]\big)-\mathbb E[k(X,y)]\;\ge\;0,\quad
\mathbb E[k(X,y)]=\det(I+\tfrac{2}{\gamma^2}\Sigma)^{-1/2}e^{-(y-\mu)^\top(\gamma^2 I+2\Sigma)^{-1}(y-\mu)},\quad
\mathbb E[k(X,X')]=\det(I+\tfrac{4}{\gamma^2}\Sigma)^{-1/2}.$$
Implemented with Cholesky solves on $\gamma^2 I + c\Sigma$ (always SPD, smallest eigenvalue $\ge\gamma^2$ —
no floor needed). A reparameterized MC estimator is provided as an optional cross-check.

**The $k(y,y)$ term.** It is the constant $1$ for this kernel and drops out of every gradient, so it is often
omitted in an implementation — but then the reported score sits at $-\tfrac12$ where a perfect prediction
should score $0$. We keep it: with it the score is half the squared MMD between $\mathcal N(\mu,\Sigma)$ and
$\delta_y$, hence $\ge 0$, and $=0$ exactly at the degenerate perfect forecast $\Sigma\to0,\ \mu\to y$.
The $\tfrac12$ is the repo convention (`src/methods/*.py`, `src/evaluation/metrics.py`), so the numbers here
are directly comparable with every other kernel-score table of the paper.

In [ ]:
def nll_chol(mu, L, y, eps):
    """Gaussian NLL with Sigma = L L^T + eps I. eps=0 uses L directly (collapse -> logdet -> -inf)."""
    d = L.shape[-1]
    r = y - mu
    Lc = L if eps == 0.0 else torch.linalg.cholesky(L @ L.T + eps * torch.eye(d, dtype=L.dtype))
    z = torch.cholesky_solve(r.T, Lc).T
    quad = (r * z).sum(-1)
    logdet = 2 * torch.log(torch.diagonal(Lc)).sum()
    return 0.5 * (quad + logdet + d * math.log(2 * math.pi))

def nll_eig(mu, Sigma, y):
    """Independent NLL path via eigendecomposition (used only for the sanity check)."""
    d = Sigma.shape[-1]
    w, Q = torch.linalg.eigh(Sigma)
    r = (y - mu) @ Q
    return 0.5 * ((r * r / w).sum(-1) + torch.log(w).sum() + d * math.log(2 * math.pi))

def kernel_score(mu, Sigma, y, h2):
    """Closed-form Gaussian kernel score of N(mu, Sigma) against y. Batched over (mu, y); Sigma global.

    score = 0.5 (k(y,y) + E[k(X,X')]) - E[k(X,y)] with k(y,y) = 1 -- the convention of the repo's
    src/methods/*.py. The constant is gradient-free, but it is what makes the score half the
    squared MMD to delta_y and hence >= 0 (a perfect forecast scores 0, not -1/2).
    """
    d = Sigma.shape[-1]; I = torch.eye(d, dtype=Sigma.dtype)
    M2 = h2 * I + 2 * Sigma; M4 = h2 * I + 4 * Sigma       # SPD by construction, smallest eig >= h2
    L2 = torch.linalg.cholesky(M2); L4 = torch.linalg.cholesky(M4)
    r = y - mu
    quad = (r * torch.cholesky_solve(r.T, L2).T).sum(-1)
    log_coef = 0.5 * d * math.log(h2)
    Ekxy = torch.exp(log_coef - torch.log(torch.diagonal(L2)).sum() - quad)
    Ekxx = torch.exp(log_coef - torch.log(torch.diagonal(L4)).sum())
    return 0.5 * (1.0 + Ekxx) - Ekxy

def kernel_score_mc(mu, Sigma, y, h2, m, gen=None):
    """Sample-based (reparameterized) kernel score -- optional cross-check of the closed form."""
    d = Sigma.shape[-1]
    Lc = torch.linalg.cholesky(Sigma + 1e-12 * torch.eye(d, dtype=Sigma.dtype))
    b = mu.shape[0]
    X  = mu.unsqueeze(1) + torch.randn(b, m, d, generator=gen) @ Lc.T
    Xp = mu.unsqueeze(1) + torch.randn(b, m, d, generator=gen) @ Lc.T
    kxy = torch.exp(-((X - y.unsqueeze(1)) ** 2).sum(-1) / h2).mean(1)
    kxx = torch.exp(-((X - Xp) ** 2).sum(-1) / h2).mean(1)
    return 0.5 * (1.0 + kxx) - kxy    # k(y, y) = 1

# the kernel-score matrices are well-conditioned by construction:
assert torch.linalg.eigvalsh(h2 * torch.eye(cfg.d) + 2 * Sigma_post).min() >= h2 - 1e-9

## 5. Model — maximally controlled

* **Mean:** a single linear layer $\mathbb R^n\!\to\!\mathbb R^d$. The true posterior mean is linear, so
  approximation error is **zero by construction**; with `oracle_mean` we initialize at the closed-form
  $A$ to isolate the covariance dynamics.
* **Covariance:** one **fixed global parameter** (the true posterior covariance is constant).
  `full` = unconstrained lower-triangular $L$ with softplus diagonal; `diagonal` = softplus $\sigma_i$.

In [ ]:
def softplus_inv(y):
    return y + torch.log(-torch.expm1(-y))

class GaussModel(torch.nn.Module):
    def __init__(self, cfg, Sigma_init, A_init=None, cov_class="full"):
        super().__init__()
        d, n = cfg.d, cfg.n
        self.cov_class = cov_class; self.offset = 1e-6
        self.mean = torch.nn.Linear(n, d, bias=True).double()
        if A_init is not None:
            with torch.no_grad():
                self.mean.weight.copy_(A_init); self.mean.bias.zero_()
        if cov_class == "full":
            L0 = torch.linalg.cholesky(Sigma_init + 1e-9 * torch.eye(d))
            raw = L0.clone()
            raw[range(d), range(d)] = softplus_inv(torch.diagonal(L0).clamp_min(1e-6) - self.offset).clamp_min(-20)
            self.L_raw = torch.nn.Parameter(raw)
        else:
            self.s_raw = torch.nn.Parameter(softplus_inv(torch.diagonal(Sigma_init).clamp_min(1e-6).sqrt()))
    def L(self):
        d = self.L_raw.shape[0]
        diag = torch.nn.functional.softplus(torch.diagonal(self.L_raw)) + self.offset
        return torch.tril(self.L_raw, -1) + torch.diag(diag)
    def sigma_diag(self):
        return torch.nn.functional.softplus(self.s_raw) + self.offset
    def Sigma(self):
        if self.cov_class == "full":
            L = self.L(); return L @ L.T
        return torch.diag(self.sigma_diag() ** 2)
    def cov_params(self):
        return [self.L_raw] if self.cov_class == "full" else [self.s_raw]
    def mean_params(self):
        return list(self.mean.parameters())
    def forward(self, x_obs):
        return self.mean(x_obs)

### Population objectives and the kernel-score minimizer $\Sigma_{\mathrm{KS}}^\star$

Because the truth is Gaussian, both population objectives are closed form (mean is exact):
$$\mathcal L^{\mathrm{NLL}}_{\mathrm{pop}}(\Sigma_\theta)=\tfrac12\!\left[\mathrm{tr}(\Sigma_\theta^{-1}\Sigma_{\mathrm{post}})+\log\det(2\pi\Sigma_\theta)\right],\qquad
\mathcal L^{\mathrm{KS}}_{\mathrm{pop}}(\Sigma_\theta)=\tfrac12\Big[1+\det\!\big(I+\tfrac{4}{\gamma^2}\Sigma_\theta\big)^{-1/2}\Big]-\det\!\big(I+\tfrac{2}{\gamma^2}(\Sigma_\theta+\Sigma_{\mathrm{post}})\big)^{-1/2}.$$
$\Sigma_{\mathrm{KS}}^\star=\arg\min \mathcal L^{\mathrm{KS}}_{\mathrm{pop}}$ (found by L-BFGS from several inits, cached)
makes the $h$-dependent bias explicit so it is not mistaken for failure.

In [ ]:
def pop_nll(Sigma_theta, Sigma_post):
    d = Sigma_theta.shape[0]
    Lc = torch.linalg.cholesky(Sigma_theta)
    logdet = 2 * torch.log(torch.diagonal(Lc)).sum()
    tr = torch.trace(torch.cholesky_solve(Sigma_post, Lc))
    return 0.5 * (tr + logdet + d * math.log(2 * math.pi))

def _logdet_I_plus(c, Sigma):
    return torch.linalg.slogdet(torch.eye(Sigma.shape[0], dtype=Sigma.dtype) + c * Sigma)[1]

def pop_kernel_score(Sigma_theta, Sigma_post, h2):    # mean exact; same 0.5 (1 + .) - . convention
    t1 = torch.exp(-0.5 * _logdet_I_plus(2.0 / h2, Sigma_theta + Sigma_post))
    t2 = torch.exp(-0.5 * _logdet_I_plus(4.0 / h2, Sigma_theta))
    return 0.5 * (1.0 + t2) - t1

def compute_ks_star(Sigma_post, h2, cfg, inits=3, iters=200):
    d = cfg.d; best = None; best_val = math.inf
    for t in range(inits):
        torch.manual_seed(100 + t)
        if t == 0:
            # PD-safe init even for a degenerate Sigma_post (e.g. above Nyquist): floor by -lambda_min
            lam_min = torch.linalg.eigvalsh(Sigma_post)[0].item()
            base = Sigma_post + max(1e-3, 1e-6 - lam_min) * torch.eye(d)
        else:
            base = (0.2 + 0.8 * torch.rand(1)).item() * torch.eye(d)
        Lp = torch.linalg.cholesky(base).clone().requires_grad_(True)
        opt = torch.optim.LBFGS([Lp], max_iter=iters, line_search_fn="strong_wolfe")
        def closure():
            opt.zero_grad()
            loss = pop_kernel_score(Lp @ Lp.T, Sigma_post, h2)
            loss.backward()
            if Lp.grad is not None: Lp.grad = Lp.grad.contiguous()
            return loss
        opt.step(closure)
        with torch.no_grad():
            St = Lp @ Lp.T; val = pop_kernel_score(St, Sigma_post, h2).item()
        if val < best_val: best_val, best = val, St.detach().clone()
    return best, best_val

## 6. Training loop with diagnostics

Same optimizer / schedule / seed for every arm: Adam with `lr_mean` on the mean head and `lr` on the
covariance parameters, cosine-annealed to zero over `n_steps`.

**Why the mean head needs its own learning rate.** With `oracle_mean = False` the mean is *learned*, and
the log score couples it to the covariance: a lagging mean leaves a large residual, the NLL inflates
$\Sigma_\theta$ to absorb it, and the inflated $\Sigma_\theta$ then damps the $\Sigma_\theta^{-1}$-weighted
mean gradient — so the mean lags further. At a single shared $10^{-3}$ the NLL arms never escape that
basin (val mean RMSE $0.52$ vs. the kernel score's $0.027$, $\operatorname{tr}\Sigma_\theta = 61$ against a
true $14.9$), and training *longer* makes it worse. This is an optimization pathology of the log score,
not the collapse itself, and reporting it as collapse would overstate the case — so we tune it away.
`lr_mean` $= 3\times10^{-2}$ with cosine decay fixes the mean for both losses (RMSE $0.024$, matching the
kernel score) and leaves the covariance behaviour, which is what this experiment is about, on its own.
Everything else we ablated — mean init (default / observations-scattered / zero), covariance init scale,
covariance lr, mean-only warm-up, $3$k–$10$k steps — is neutral or harmful.

Every `log_every` steps we log the eigenvalues of
$\Sigma_\theta$, their maxima projected onto the **true-null** vs **true-signal** eigenspaces, the
(pre-clip) covariance gradient norm, the train loss, the exact **population objective**, and
$\lVert\Sigma_\theta-\Sigma_{\mathrm{post}}\rVert_F$, $\lVert\Sigma_\theta-\Sigma_{\mathrm{KS}}^\star\rVert_F$.
Non-finite losses are caught: we record `diverged_at` and continue, so a diverging arm never kills the
notebook.

In [ ]:
def train_arm(cfg, data, Sigma_pr, Sigma_post, A, arm, h2, V_null, V_sig, Sigma_KS,
              n_steps=None, seed=None):
    n_steps = n_steps or cfg.n_steps
    seed = cfg.seed if seed is None else seed
    Xobs, Y = data
    kind, cov_class = arm["kind"], arm["cov_class"]
    eps = float(arm.get("eps", 0.0))
    torch.manual_seed(seed)
    Sigma_init = torch.eye(cfg.d) * max(Sigma_post.diag().mean().item(), 1e-3)   # well-conditioned init
    model = GaussModel(cfg, Sigma_init, A_init=(A if cfg.oracle_mean else None), cov_class=cov_class)
    # Two param groups: the mean head gets its own, larger lr. Under the NLL a lagging mean
    # inflates Sigma to absorb the residual, and the inflated Sigma then damps the Sigma^{-1}-
    # weighted mean gradient -- a self-sustaining stall that a single shared lr never escapes
    # (see sec. 6). Cosine annealing on *both* groups; identical schedule for every arm.
    opt = torch.optim.Adam([{"params": model.mean_params(), "lr": cfg.lr_mean},
                            {"params": model.cov_params(),  "lr": cfg.lr}])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_steps)
    N = Xobs.shape[0]; g = torch.Generator().manual_seed(seed + 1)
    keys = ["step", "loss", "pop", "frob_post", "frob_ks", "gradnorm", "eig_null", "eig_sig", "evals"]
    logs = {kk: [] for kk in keys}; diverged_at = None
    for step in range(n_steps):
        idx = torch.randint(0, N, (cfg.batch_size,), generator=g)
        mu = model(Xobs[idx]); yb = Y[idx]
        try:
            if kind == "nll":
                L = model.L() if cov_class == "full" else torch.diag(model.sigma_diag())
                loss = nll_chol(mu, L, yb, eps).mean()
            else:
                loss = kernel_score(mu, model.Sigma(), yb, h2).mean()
            if not torch.isfinite(loss): raise FloatingPointError("non-finite loss")
            opt.zero_grad(); loss.backward()
            gnorm = torch.sqrt(sum((p.grad ** 2).sum() for p in model.cov_params() if p.grad is not None))
            if not torch.isfinite(gnorm): raise FloatingPointError("non-finite grad")
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            opt.step(); sched.step()
        except (FloatingPointError, torch._C._LinAlgError):
            diverged_at = step; break
        if step % cfg.log_every == 0:
            with torch.no_grad():
                St = model.Sigma() + (eps if kind == "nll" else 0.0) * torch.eye(cfg.d)
                w = torch.linalg.eigvalsh(St)
                en = V_null.T @ St @ V_null; es = V_sig.T @ St @ V_sig
                try:
                    pop = (pop_nll(St, Sigma_post) if kind == "nll"
                           else pop_kernel_score(St, Sigma_post, h2)).item()
                except torch._C._LinAlgError:
                    pop = float("nan")
                logs["step"].append(step); logs["loss"].append(float(loss))
                logs["pop"].append(pop)
                logs["frob_post"].append(torch.linalg.norm(St - Sigma_post).item())
                logs["frob_ks"].append(torch.linalg.norm(St - Sigma_KS).item())
                logs["gradnorm"].append(float(gnorm))
                logs["eig_null"].append(torch.linalg.eigvalsh(en).max().item() if en.numel() else 0.0)
                logs["eig_sig"].append(torch.linalg.eigvalsh(es).max().item() if es.numel() else 0.0)
                logs["evals"].append(w.cpu().numpy())
    return {"model": model, "logs": logs, "diverged_at": diverged_at, "arm": arm,
            "Sigma_final": (model.Sigma() + (eps if kind == "nll" else 0.0) * torch.eye(cfg.d)).detach()}

## 7. Run the arms, looped over the band limit $B$

We repeat the *entire* pipeline for four band limits $B\in\{4,16,32,64\}$ (active modes $s=2B+1$,
observations $n=d/k$ fixed). Each $B$ sits in a different regime relative to the Nyquist point $n/s=1$,
so the true posterior rank deficiency — and hence the severity of the NLL collapse — changes row to row.
For every $B$ we train all arms
$\{\text{kernel score},\ \text{NLL}(\varepsilon)\ \text{for }\varepsilon\in\text{eps\_list},\ \text{NLL raw}\}\times\{\text{full},\text{diagonal}\}$
with the shared optimizer / schedule / seed.

In [ ]:
def arm_name(arm):
    if arm["kind"] == "ks": return "KS"
    return "NLL raw" if float(arm.get("eps", 0.0)) == 0.0 else f"NLL eps={arm['eps']:.0e}"

def make_arms(cov_class, eps_list):
    arms = [{"kind": "ks", "cov_class": cov_class}]
    arms += [{"kind": "nll", "eps": e, "cov_class": cov_class} for e in eps_list]
    arms += [{"kind": "nll", "eps": 0.0, "cov_class": cov_class}]
    return arms


def build_problem(cfg):
    """Everything B-dependent: prior, data, closed-form posterior, bandwidth, Sigma_KS*."""
    torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
    Sigma_pr, P = build_prior(cfg)
    S, obs_idx = selection(cfg)
    _, xobs_tr, y_tr = make_dataset(cfg, Sigma_pr, P, cfg.n_train, cfg.seed + 10)
    x_va, xobs_va, y_va = make_dataset(cfg, Sigma_pr, P, cfg.n_val, cfg.seed + 20)
    A_true, Sigma_post = posterior(cfg, Sigma_pr, S)
    evals_post, _, V_null, V_sig = eig_partition(Sigma_post)
    rank_post = int((evals_post > 1e-8 * max(float(evals_post.max()), 1.0)).sum())
    regime = "above" if cfg.n > cfg.s else ("at" if cfg.n == cfg.s else "below")
    h2 = bandwidth(cfg, Sigma_post, Sigma_pr)
    Sigma_KS, _ = compute_ks_star(Sigma_post, h2, cfg)
    tag = f"n/s = {cfg.n/cfg.s:.2f}, {regime} Nyquist, eff. rank(Sigma_post) = {rank_post}/{cfg.d}"
    return dict(cfg=cfg, Sigma_pr=Sigma_pr, S=S, obs_idx=obs_idx,
                x_va=x_va, xobs_va=xobs_va, y_va=y_va, xobs_tr=xobs_tr, y_tr=y_tr,
                A_true=A_true, Sigma_post=Sigma_post, V_null=V_null, V_sig=V_sig,
                rank_post=rank_post, regime=regime, h2=h2, Sigma_KS=Sigma_KS,
                REGIME_TAG=tag)

runs = {}
t0 = time.time()
for B in B_values:
    cfg_B = replace(cfg, B=B)
    prob = build_problem(cfg_B)
    print(f"\n=== B={B}:  s=2B+1={cfg_B.s}, n={cfg_B.n},  {prob['REGIME_TAG']},  h2={prob['h2']:.3f} ===")
    results = {"full": {}, "diagonal": {}}
    data_tr = (prob["xobs_tr"], prob["y_tr"])
    for cov_class in ("full",):
        for arm in make_arms(cov_class, cfg_B.eps_list):
            res = train_arm(cfg_B, data_tr, prob["Sigma_pr"], prob["Sigma_post"],
                            prob["A_true"], arm, prob["h2"], prob["V_null"],
                            prob["V_sig"], prob["Sigma_KS"])
            results[cov_class][arm_name(arm)] = res
            lg = res["logs"]
            print(f"  [{cov_class:8s}] {arm_name(arm):14s} "
                  f"frob_post={lg['frob_post'][-1]:8.3f}  "
                  f"final_gradnorm={lg['gradnorm'][-1]:.2e}  diverged_at={res['diverged_at']}")
    prob["results"] = results
    runs[B] = prob
print(f"\nall B x arms trained in {time.time()-t0:.1f}s")

### Fig. 1b — Predicted mean ± 2σ per method, one row per band limit $B$
Same view as the *example data* figure above. The **leftmost column** is the closed-form truth
$\mu_{\mathrm{post}} \pm 2\sigma_{\mathrm{post}}$; the remaining columns show each **arm's learned**
$\mu_\theta \pm 2\sigma_\theta$ (kernel score, NLL $\varepsilon=10^{-6}$, raw NLL). **Rows** are the four band
limits $B\in\{4,16,32,64\}$. Reading down a column shows how the NLL bands blow up / distort more severely as
the rank deficiency grows, while the kernel-score band keeps tracking the signal. We render this once for
the **full** covariance (below) and once for the **diagonal** covariance.

In [ ]:
# Fig. 1b -- predicted mean +/- 2 sigma;  rows = band limit B, cols = truth + method  (full covariance)
i_ex       = 1                                             # which validation example to display
cov_kind   = "full"
panel_arms = ["KS", "NLL eps=1e-06", "NLL raw"]
col_names  = ["truth"] + panel_arms                        # leftmost column = closed-form posterior
TRUTH_COL  = COLORS["truth"]
grid       = torch.arange(cfg.d) / cfg.d                   # physical domain [0, 1)

# double column; per-panel height/width 0.62, so the grid scales with the number of panels
fig, axes = pc.subplots(len(B_values), len(col_names),
                        aspect=0.5 * len(B_values) / len(col_names), sharex=True, sharey=True)
for r, B in enumerate(B_values):
    prob = runs[B]; results = prob["results"]
    xi, xobsi, obs_idx = prob["x_va"][i_ex], prob["xobs_va"][i_ex], prob["obs_idx"]
    for c, name in enumerate(col_names):
        ax = axes[r, c]
        if name == "truth":
            with torch.no_grad():
                mu = prob["A_true"] @ xobsi
            sd  = prob["Sigma_post"].diag().clamp_min(0).sqrt()
            col, dv = TRUTH_COL, None
            title = LABELS["truth"]
        else:
            res = results[cov_kind][name]
            with torch.no_grad():
                mu = res["model"](xobsi)
            sd  = res["Sigma_final"].diag().clamp_min(0).sqrt()
            col, dv = arm_color(name), res["diverged_at"]
            title = arm_label(name)
        ax.fill_between(grid, (mu - 2 * sd), (mu + 2 * sd), color=col, alpha=PLOT.band_alpha)
        ax.plot(grid, xi, color=COLORS["signal"], label=LABELS["signal"])
        ax.plot(grid, mu, "--", color=col)
        ax.plot(obs_idx / cfg.d, xobsi, "o", color=COLORS["observation"], label=LABELS["observation"])
        ax.set_ylim([-1.5, 1.5])
        ax.set_xlim(0, 1)
        if r == 0:
            ax.set_title(title + (f"\n(diverged @ {dv})" if dv is not None else ""))
        elif dv is not None:
            ax.set_title(f"diverged @ {dv}", fontsize=PLOT.annotation_size)
        if c == 0:
            ax.set_ylabel(f"B = {B}")
# only the signal and its observations are labeled; the band/mean color is the column title
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncol=2)
pc.save(fig, "toy1_full.pdf")
plt.show()

### Fig. 3 — Covariance-gradient norms (full covariance) per band limit $B$
One panel per $B$. The log-score gradient magnitude scales like $1/\varepsilon$ as the floor shrinks
(ill-posed limit) and grows with the rank deficiency; the kernel-score gradient stays bounded (intrinsic
$h^2$ floor). We show the **full** covariance arms only.

In [ ]:
# Fig. 3 -- covariance-gradient norms, one panel per B (FULL covariance arms only)
fig, axes = pc.subplots(1, len(B_values), aspect=0.95 / len(B_values))   # double column
for ax, B in zip(axes, B_values):
    results = runs[B]["results"]
    for name, res in results["full"].items():
        lg = res["logs"]
        ax.plot(lg["step"], np.clip(lg["gradnorm"], 1e-16, None),
                color=arm_color(name), label=arm_label(name))
    ax.set_yscale("log"); ax.set_xlabel("Training step")
    # the regime tag is far wider than one panel at 6.75 in, so wrap it onto a smaller second line
    ax.set_title(f"B = {B}", fontsize=PLOT.title_size)
axes[0].set_ylabel(r"$\|\nabla_{\hat{\Sigma}}\mathcal{L}\|$")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncol=len(labels))
pc.save(fig, "toy1_grad_norm.pdf")
plt.show()

#### Presentation copy — Fig. 3 as a 2 × 2 grid

The same four gradient-norm panels as above, arranged 2 × 2 instead of in one row. Written to
`figures/presentation/` (git-ignored), never over the paper figure.

In [ ]:
# Presentation copy of Fig. 3: the four band limits as a 2x2 grid instead of one row.
fig, axes = pc.subplots(2, 2, aspect=0.78, columns=4.5, sharex=True)
for ax, B in zip(axes.flat, B_values):
    results = runs[B]["results"]
    for name, res in results["full"].items():
        lg = res["logs"]
        ax.plot(lg["step"], np.clip(lg["gradnorm"], 1e-16, None),
                color=arm_color(name), label=arm_label(name))
    ax.set_yscale("log")
    ax.set_title(f"B = {B}", fontsize=PLOT.title_size)
for ax in axes[1, :]:
    ax.set_xlabel("Training step")
for ax in axes[:, 0]:
    ax.set_ylabel(r"$\|\nabla_{\hat{\Sigma}}\mathcal{L}\|$")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncol=len(labels))
pc.save(fig, pc.PRESENTATION_DIR / "toy1_grad_norm.pdf")
plt.show()


# Reduced plots

### Fig. R1 — Main-paper version of Fig. 1b: KS vs. NLL, $B\in\{4,32\}$
A smaller, two-column/two-row reduction of Fig. 1b for the main paper: only the **kernel score** and
**raw NLL** (labeled "NLL") arms, only the two band limits $B\in\{4,32\}$, full covariance. Axes are
shared across all panels.

We show $B=32$ (true rank $49/128$) rather than $B=64$: with the mean head properly tuned, $B=64$ leaves
only $16$ crushed directions out of $128$ — too mild a deficiency for the log score to suffer, and there
it in fact *beats* the kernel score on $\lVert\Sigma_\theta-\Sigma_{\mathrm{post}}\rVert_F$
($0.28$ vs. $0.66$), exactly as a proper score with a well-specified model should. The rows here are the
two regimes where the collapse actually bites.

The closed-form posterior is no longer a column of its own here: it enters every panel as the
grey $\boldsymbol{\mu}^\star \pm 2\sigma^\star$ region on the bottom layer, so each arm's band is read against the
truth it should match.

In [ ]:
# Fig. R1 -- reduced predicted-mean bands for the main paper: KS vs. NLL (raw), B in {4, 32}
i_ex         = 1
cov_kind     = "full"
B_reduced    = (4, 32)
panel_arms   = ["KS", "NLL raw"]
panel_titles = [arm_label(n) for n in panel_arms]
grid         = torch.arange(cfg.d) / cfg.d                 # physical domain [0, 1)
TRUTH_BAND_ALPHA = 0.20                                    # the truth's grey, lighter than an arm's band

# the only SINGLE-column figure of the paper -- everything else spans both columns
fig, axes = pc.subplots(len(B_reduced), len(panel_arms), aspect=0.55, columns="single",
                        sharex=True, sharey=True)
for r, B in enumerate(B_reduced):
    prob = runs[B]; results = prob["results"]
    xi, xobsi, obs_idx = prob["x_va"][i_ex], prob["xobs_va"][i_ex], prob["obs_idx"]
    # the closed-form posterior of this B: the truth column of Fig. 1b, moved underneath
    # every panel as a grey band instead of standing beside them as its own column
    mu_star = prob["A_true"] @ xobsi
    sd_star = prob["Sigma_post"].diag().clamp_min(0).sqrt()
    for c, name in enumerate(panel_arms):
        ax = axes[r, c]
        res = results[cov_kind][name]
        with torch.no_grad():
            mu = res["model"](xobsi)
        sd  = res["Sigma_final"].diag().clamp_min(0).sqrt()
        col = arm_color(name)
        # the truth sits on the bottom layer (zorder 0), so the arm's own band reads on top of it
        ax.fill_between(grid, (mu_star - 2 * sd_star), (mu_star + 2 * sd_star), color="black",
                        alpha=TRUTH_BAND_ALPHA, lw=0, zorder=0)
        ax.fill_between(grid, (mu - 2 * sd), (mu + 2 * sd), color=col, alpha=PLOT.band_alpha)
        ax.plot(grid, xi, color=COLORS["signal"], label=LABELS["signal"])
        ax.plot(grid, mu, "--", color=col)
        ax.plot(obs_idx / cfg.d, xobsi, "o", ms=0.8 * PLOT.marker_size,
                color=COLORS["observation"], label=LABELS["observation"])
        ax.set_ylim([-1.4, 1.4])
        ax.set_xlim(0, 1)
        if r == 0:
            ax.set_title(panel_titles[c])
        if c == 0:
            ax.set_ylabel(f"B = {B}")

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncol=3)
pc.save(fig, "toy1_small.pdf")
plt.show()

---
## 8. Metrics — every arm across five seeds

The figures above are one seed. Here the **whole pipeline** is rerun for `seeds = 0..4` —
each seed redraws the training and validation data, reinitializes the model and reshuffles
the minibatch order — and we report held-out mean $\pm$ 1 standard deviation over the seeds.
The table reports the single quantity this experiment is about,

$$\lVert\Sigma_\theta-\Sigma^\star\rVert_F,$$

the Frobenius distance from the learned covariance to the closed-form posterior covariance
$\Sigma_{\mathrm{post}}$ (read off the same `Sigma_final` the figures draw, i.e. including the
$\varepsilon I$ of the floored arm). Rows are the three methods, **columns the four band limits**
$B\in\{4,16,32,64\}$; lower is better.

**Why not the held-out scores.** They are still computed in `eval_metrics` (`nll`, `ks` in the
`metrics` frame) but no longer tabulated: each is the training objective of one of the arms and
so favours it by construction, and the NLL is finite here only because $\sigma_{\mathrm{obs}} =
10^{-3} > 0$ holds the truth just off the PSD boundary — in the $\sigma_{\mathrm{obs}}\to0$ limit
it is undefined, which is the point made at the top of the notebook.


In [ ]:
# ---- Metrics: every arm x 5 seeds x band limit, evaluated on the held-out validation set ----
METRIC_SEEDS = (0, 1, 2, 3, 4)
METRIC_ARMS  = ["KS", "NLL eps=1e-06", "NLL raw"]
METRIC_KEYS  = ("frob", "nll", "ks")
arms_by_name = {arm_name(a): a for a in make_arms("full", cfg.eps_list)}

def eval_metrics(res, prob):
    """Held-out metrics of one trained arm (all lower-is-better)."""
    with torch.no_grad():
        Sigma_theta = res["Sigma_final"]              # includes the eps I of a floored arm
        mu = res["model"](prob["xobs_va"])
        y  = prob["y_va"]
        out = {"frob": torch.linalg.norm(Sigma_theta - prob["Sigma_post"]).item()}
        # Sigma_theta is PD by construction (softplus diagonal + offset), but an arm that
        # blew up mid-training can leave it unusable -- report NaN rather than crash the sweep.
        try:
            out["nll"] = nll_chol(mu, torch.linalg.cholesky(Sigma_theta), y, 0.0).mean().item()
            out["ks"]  = kernel_score(mu, Sigma_theta, y, prob["h2"]).mean().item()
        except torch._C._LinAlgError:
            out["nll"] = out["ks"] = float("nan")
    return out

records, t0 = [], time.time()
for B in B_values:
    for seed in METRIC_SEEDS:
        # a new seed redraws the data *and* reinitializes the model and the batch order
        prob = build_problem(replace(cfg, B=B, seed=seed))
        for name in METRIC_ARMS:
            res = train_arm(prob["cfg"], (prob["xobs_tr"], prob["y_tr"]), prob["Sigma_pr"],
                            prob["Sigma_post"], prob["A_true"], arms_by_name[name],
                            prob["h2"], prob["V_null"], prob["V_sig"], prob["Sigma_KS"],
                            seed=seed)
            records.append({"B": B, "seed": seed, "method": name,
                            "diverged_at": res["diverged_at"], **eval_metrics(res, prob)})
    print(f"B = {B:2d}:  {len(METRIC_SEEDS)} seeds x {len(METRIC_ARMS)} arms   "
          f"({time.time() - t0:6.1f}s elapsed)")

metrics = pd.DataFrame.from_records(records)
print(f"\n{len(metrics)} runs, {int(metrics['diverged_at'].notna().sum())} diverged")


In [ ]:
# ---- one table: rows = methods, columns = band limit B, mean +/- std over seeds ----
# Only the Frobenius distance to Sigma_post is tabulated; the held-out NLL / kernel score stay in
# `metrics` (each is one arm's own training objective, so neither is a neutral comparison).
def _fmt(mean, std):
    return "nan" if not np.isfinite(mean) else f"{mean:.3g} $\\pm$ {std:.2g}"

agg = metrics.groupby(["B", "method"])[list(METRIC_KEYS)].agg(["mean", "std"])

def frob_table(index, col_fmt="B = {B}"):
    """rows = methods, columns = band limit B; entries ||Sigma_theta - Sigma*||_F."""
    return pd.DataFrame(
        {col_fmt.format(B=B): [_fmt(agg.loc[(B, m), ("frob", "mean")],
                                    agg.loc[(B, m), ("frob", "std")]) for m in METRIC_ARMS]
         for B in B_values},
        index=pd.Index(index, name="Method"))

print(f"||Sigma_theta - Sigma*||_F   (mean +/- std over {len(METRIC_SEEDS)} seeds, lower is better)")
for B in B_values:
    print(f"  B = {B:2d}:  {runs[B]['REGIME_TAG']}")
print()
print(frob_table(METRIC_ARMS).to_string().replace("$\\pm$", "+/-"))


In [ ]:
# ---- and the same numbers as LaTeX, ready to paste into the paper ----
# REGIME_TAG is a plain-text label (it has bare underscores), so the caption is rebuilt
# in math mode here rather than reused -- `escape=False` would hand it straight to LaTeX.
regimes = "; ".join(
    f"$B = {B}$: $n/s = {runs[B]['cfg'].n / runs[B]['cfg'].s:.2f}$, "
    f"$\\operatorname{{rank}} \\Sigma^\\star = {runs[B]['rank_post']}/{cfg.d}$"
    for B in B_values)
caption = (f"Frobenius distance $\\lVert\\Sigma_\\theta-\\Sigma^\\star\\rVert_F$ of the learned covariance "
           f"to the true posterior covariance, per band limit $B$ ({regimes}). "
           f"Mean $\\pm$ std over {len(METRIC_SEEDS)} seeds; lower is better.")
print(frob_table([arm_label(m) for m in METRIC_ARMS], col_fmt=r"$B = {B}$").rename_axis(None).to_latex(
    escape=False, column_format="l" + "c" * len(B_values),
    caption=caption, label="tab:toy1_metrics"))
